In [1]:
!pip install faiss-gpu-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 30.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetA.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetA.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-A/")

!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetB_originalSplit.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetB_originalSplit.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-B/")

!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetC.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetC.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-C/")



     29,425,120 100%   26.90MB/s    0:00:01 (xfr#1, to-chk=0/1)
     67,890,739 100%   83.83MB/s    0:00:00 (xfr#1, to-chk=0/1)
    101,143,907 100%   45.68MB/s    0:00:02 (xfr#1, to-chk=0/1)


In [14]:
#load the modules
import keras
from keras import models, layers
from keras.activations import relu, softmax
from keras.applications import VGG19, VGG16 #trying VGG16 instead
from keras.models import Sequential, load_model, Model
from keras.optimizers import Adam, SGD
from keras.callbacks import ModelCheckpoint, Callback, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.layers import Activation, Dropout, Dense, Flatten, GlobalAveragePooling2D
import matplotlib.pyplot as plt
from tensorflow.keras.losses import CategoricalFocalCrossentropy
import sys
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
sys.modules['Image'] = Image

seed = 3434343
keras.utils.set_random_seed(seed)

In [15]:
filepath = "/content/drive/MyDrive/CNN_benchmark/FINAL_kaakaa_drop0.48_3434343/saved_model.h5"

model=load_model(filepath)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,78

 Total params: 26,451,282 (100.90 MB)

 Trainable params: 26,451,280 (100.90 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [16]:
#per https://medium.com/vector-database/how-to-get-the-right-vector-embeddings-83295ced7f35

#BEHEAD HIM!!!!!
x = model.layers[-2].output
model = Model(inputs = model.input, outputs = x)

model.trainable = False #we already trained it

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,78

 Total params: 26,447,168 (100.89 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 26,447,168 (100.89 MB)

In [17]:
# Keras' data generator can be used to pass the images through the convolutional neural network and apply
#rotation and zoom transformations to the images. Check https://keras.io/preprocessing/image/ for more transformations
train_data = ImageDataGenerator(
        preprocessing_function=keras.applications.vgg19.preprocess_input)

A_generator = train_data.flow_from_directory(
        directory=r"./dataset-A/train",
        target_size=(224, 224),
        batch_size=8,
        shuffle=False)

B_generator = train_data.flow_from_directory(
        directory=r"./dataset-B/train",
        target_size=(224, 224),
        batch_size=8,
        shuffle=False)

C_generator = train_data.flow_from_directory(
        directory=r"./dataset-C/train",
        target_size=(224, 224),
        batch_size=8,
        shuffle=False)

Found 807 images belonging to 8 classes.
Found 919 images belonging to 16 classes.
Found 4044 images belonging to 17 classes.


In [ ]:
from faiss import normalize_L2

def add_to_embeddings_dict(train_generator, id):
  train_generator.reset()

  invert_traingen = {v: k for k, v in train_generator.class_indices.items()}
  print(invert_traingen)
  embeddings = []
  org_id = 0
  steps = len(train_generator) #should = 1000 with 8 per batch and 8000 images
  print("steps", steps)

  for step in range(steps):
    x_batch, y_batch=next(train_generator) #getting our batch

    for i in range(len(x_batch)):
      #print(f"{id}, batch indices {batch_indices[i]}, filepath {train_generator.filepaths[batch_indices[i]]}")
      path = train_generator.filepaths[org_id]
      image=np.expand_dims(x_batch[i], axis=0)
      label = np.where(y_batch[i] == np.amax(y_batch[i]))[0][0]
      vector=model.predict(image, verbose=0)
      #vectors need to be normalized both before adding to the index and before searching
      normalize_L2(vector)

      if invert_traingen[label] not in train_generator.filepaths[org_id]:
        print(f"Error! {invert_traingen[label]} is not in {train_generator.filepaths[org_id]}; there's a file mis-match. Have you checked your indexing?")
      embeddings.append({'id': id + org_id, 'label':invert_traingen[label], 'path': path, 'vector':vector})
      org_id += 1

  return embeddings, id+org_id

A_embeddings, id = add_to_embeddings_dict(A_generator, 0)
B_embeddings, id = add_to_embeddings_dict(B_generator, id)
C_embeddings, id = add_to_embeddings_dict(C_generator, id)

{0: '-B', 1: 'L-MB', 2: 'LM-G', 3: 'MR-R', 4: 'O-RS', 5: 'WS-P', 6: 'Y-GM', 7: 'YM-Y'}
steps 101
{0: '-Y', 1: 'B-', 2: 'K-WP', 3: 'L-MB', 4: 'L-XX', 5: 'L-YM', 6: 'LM-G', 7: 'MO-R', 8: 'MR-L', 9: 'O-LW', 10: 'P-XX', 11: 'PR-K', 12: 'W-BB', 13: 'WX-P', 14: 'Y-GL', 15: 'YM-Y'}
steps 115
{0: '-B', 1: '-Y', 2: 'B-', 3: 'K-PW', 4: 'KR-L', 5: 'L-MB', 6: 'LM-G', 7: 'MO-R', 8: 'O-LW', 9: 'OL-Y', 10: 'RM-X', 11: 'W-GL', 12: 'WX-P', 13: 'X-XX', 14: 'XX-O', 15: 'Y-GL', 16: 'YM-Y'}
steps 506


In [ ]:
all_embeddings = A_embeddings + B_embeddings + C_embeddings

In [ ]:
#now for the scary part..
#https://towardsdatascience.com/building-an-image-similarity-search-engine-with-faiss-and-clip-2211126d08fa/
import faiss

#i decided to change their image paths code to do a json dictionary as i'm familiar with parsing those
#and django/react loves them
import json

def create_faiss_index(embeddings, output_path):

    first_vect = embeddings[0]['vector']

    #converting back to np.array
    #did this bc json doesn't serialize np arrays
    dimension = np.array(first_vect).shape[1]
    print(f"Dimension: {dimension}")
    index = faiss.IndexFlatIP(dimension) #inner product index
    index = faiss.IndexIDMap(index)

    #flattening to 2d.... this is a mess due to my list of vectors but stay with me
    vectors = np.vstack([item['vector'] for item in embeddings]).astype('float32')
    #also makes all of them np.arrays

    #removing the vectors... not terribly efficient but I'm simply testing things out
    for item in embeddings:
      item.pop('vector', None)

    # Add vectors to the index with IDs
    index.add_with_ids(vectors, np.array(range(len(embeddings))))

    # Save the index
    faiss.write_index(index, output_path)
    print(f"Index created and saved to {output_path}")

    # Save image paths
    with open(output_path + '.paths.json', 'w') as f:
      json.dump(embeddings, f)

    return index

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index = create_faiss_index(all_embeddings, OUTPUT_INDEX_PATH)

Dimension: 256
Index created and saved to /content/vector.index


In [ ]:
!cp vector.index.paths.json /content/drive/MyDrive/FAISS_index/CNN/Dataset_ALL/
!cp vector.index /content/drive/MyDrive/FAISS_index/CNN/Dataset_ALL/

In [3]:
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/CNN/Dataset_ALL/vector.index.paths.json" "./"
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/CNN/Dataset_ALL/vector.index" "./"

        451,603 100%  199.72MB/s    0:00:00 (xfr#1, to-chk=0/1)
      5,954,730 100%   52.29MB/s    0:00:00 (xfr#1, to-chk=0/1)


In [4]:
import faiss
import json

#https://towardsdatascience.com/building-an-image-similarity-search-engine-with-faiss-and-clip-2211126d08fa/ again
def load_faiss_index(index_path):
    index = faiss.read_index(index_path)
    with open(index_path + '.paths.json', 'r') as f:
        image_paths = json.load(f)
    print(f"Index loaded from {index_path}")
    return index, image_paths

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index, embeddings = load_faiss_index(OUTPUT_INDEX_PATH)



Index loaded from /content/vector.index


In [11]:
#we can now try to predict based on our validation set...
#https://medium.com/@devbytes/similarity-search-with-faiss-a-practical-guide-to-efficient-indexing-and-retrieval-e99dd0e55e8c
from collections import Counter
import statistics
from faiss import normalize_L2

def majority_voting_cosine(faiss_index, embeddings, generator):
  x_batch, y_batch=next(generator)

  correct_guesses = []
  correct_distances = []
  incorrect_distances = []
  incorrect_guesses = []
  all_distances = []
  new_birds = []

  for i in range(len(x_batch)):
    print(i)
    image=np.expand_dims(x_batch[i], axis=0)
    query_vector=model.predict(image, verbose=0)

    normalize_L2(query_vector)
    k = 5  # Number of nearest neighbors to retrieve

    distances, indices = faiss_index.search(query_vector, k)

    #our key for int->label
    key = {v: k for k, v in generator.class_indices.items()}

    label = key[np.where(y_batch[i] == np.amax(y_batch[i]))[0][0]]

    print(f"true label: {label}")

    all_guesses = []
    neighbor_distances = []

    for i, index in enumerate(indices[0]):
      distance = distances[0][i]
      all_distances.append(distance)
      neighbor_distances.append(distance)

      #kind of a mess. dealing with the json dictionary
      id = embeddings[index]['id']
      new_label = embeddings[index]['label']
      all_guesses.append(new_label)
      print(f"Nearest neighbor {i+1}: {embeddings[index]}, Distance {distance}")

    majority_vote = Counter(all_guesses)
    winner = sorted(all_guesses, key=lambda x: majority_vote[x], reverse=True)[0]
    if winner == label:
      correct_guesses.append(winner)
      correct_distances.extend(neighbor_distances)
    else: #if it's the wrong label
      incorrect_guesses.append(winner)
      incorrect_distances.extend(neighbor_distances)
      if majority_vote[winner] <= k/2 or statistics.median(neighbor_distances) < 0.8:
        print(f"Potential new bird detected - true label {label}")
        new_birds.append(label)



  print(f"Total accuracy: {len(correct_guesses)/(len(correct_guesses)+len(incorrect_guesses))}")

  print(f"Median of all distances: {statistics.median(all_distances)}")

  print(f"Median distance of incorrect guesses: {statistics.median(incorrect_distances)}")

  print(f"Median distance of correct guesses: {statistics.median(correct_distances)}")

  print(f"Lowest: {min(all_distances)} highest: {max(all_distances)}")

  if len(new_birds) > 0:

    print(f"Potential new birds: {new_birds}")

    actual_new_birds = [bird for bird in new_birds if bird not in generator.class_indices.keys()]

    print(f"Of the new birds detected, the following were actually new (not in Dataset B): {actual_new_birds}")

In [18]:
def run_test_on_dir(directory):
  from pathlib import Path
  r_directory = Path("./dataset-A/val")

  num_files = sum(1 for f in r_directory.rglob("*") if f.is_file())
  print(num_files)

  test_data = ImageDataGenerator(
        preprocessing_function=keras.applications.vgg19.preprocess_input)

  test_generator = test_data.flow_from_directory(
          directory=directory,
          target_size=(224, 224),
          batch_size=num_files,
          shuffle=True)

  majority_voting_cosine(faiss_index, embeddings, test_generator)

In [19]:
run_test_on_dir("./dataset-A/val")

176
Found 176 images belonging to 8 classes.
0
true label: L-MB
Nearest neighbor 1: {'id': 207, 'label': 'L-MB', 'path': './dataset-A/train/L-MB/GH015978_22.jpg'}, Distance 0.9688141942024231
Nearest neighbor 2: {'id': 199, 'label': 'L-MB', 'path': './dataset-A/train/L-MB/GH015978_15.jpg'}, Distance 0.9646557569503784
Nearest neighbor 3: {'id': 205, 'label': 'L-MB', 'path': './dataset-A/train/L-MB/GH015978_20.jpg'}, Distance 0.9606152772903442
Nearest neighbor 4: {'id': 217, 'label': 'L-MB', 'path': './dataset-A/train/L-MB/GH015978_8.jpg'}, Distance 0.959673285484314
Nearest neighbor 5: {'id': 183, 'label': 'L-MB', 'path': './dataset-A/train/L-MB/GH015974_6.jpg'}, Distance 0.9593083262443542
1
true label: WS-P
Nearest neighbor 1: {'id': 684, 'label': 'WS-P', 'path': './dataset-A/train/WS-P/GH016354_1.jpg'}, Distance 0.9414420127868652
Nearest neighbor 2: {'id': 658, 'label': 'WS-P', 'path': './dataset-A/train/WS-P/GH016346_0.jpg'}, Distance 0.9238321781158447
Nearest neighbor 3: {'id':

In [20]:
run_test_on_dir("./dataset-B/val")

176
Found 2063 images belonging to 16 classes.
0
true label: WX-P
Nearest neighbor 1: {'id': 1618, 'label': 'WX-P', 'path': './dataset-B/train/WX-P/GH013569_0.jpg'}, Distance 0.9714198112487793
Nearest neighbor 2: {'id': 1583, 'label': 'WX-P', 'path': './dataset-B/train/WX-P/GH013547_0.jpg'}, Distance 0.9680447578430176
Nearest neighbor 3: {'id': 1616, 'label': 'WX-P', 'path': './dataset-B/train/WX-P/GH013568_1.jpg'}, Distance 0.9663978219032288
Nearest neighbor 4: {'id': 1613, 'label': 'WX-P', 'path': './dataset-B/train/WX-P/GH013566_2.jpg'}, Distance 0.96622633934021
Nearest neighbor 5: {'id': 1622, 'label': 'WX-P', 'path': './dataset-B/train/WX-P/GH013570_0.jpg'}, Distance 0.9661296606063843
1
true label: -Y
Nearest neighbor 1: {'id': 843, 'label': '-Y', 'path': './dataset-B/train/-Y/GH015251_0.jpg'}, Distance 0.8076812028884888
Nearest neighbor 2: {'id': 857, 'label': '-Y', 'path': './dataset-B/train/-Y/GH015271_0.jpg'}, Distance 0.8022893667221069
Nearest neighbor 3: {'id': 902, '

In [21]:
run_test_on_dir("./dataset-C/val")

176
Found 1174 images belonging to 17 classes.
0
true label: W-GL
Nearest neighbor 1: {'id': 4751, 'label': 'W-GL', 'path': './dataset-C/train/W-GL/GX012240_3.jpg'}, Distance 0.9247556924819946
Nearest neighbor 2: {'id': 4752, 'label': 'W-GL', 'path': './dataset-C/train/W-GL/GX012240_4.jpg'}, Distance 0.919219970703125
Nearest neighbor 3: {'id': 4736, 'label': 'W-GL', 'path': './dataset-C/train/W-GL/GX012234_0.jpg'}, Distance 0.9082988500595093
Nearest neighbor 4: {'id': 4761, 'label': 'W-GL', 'path': './dataset-C/train/W-GL/GX012243_3.jpg'}, Distance 0.908099353313446
Nearest neighbor 5: {'id': 4760, 'label': 'W-GL', 'path': './dataset-C/train/W-GL/GX012243_2.jpg'}, Distance 0.9057126641273499
1
true label: B-
Nearest neighbor 1: {'id': 3548, 'label': 'B-', 'path': './dataset-C/train/B-/GX014132_12.jpg'}, Distance 0.9564865231513977
Nearest neighbor 2: {'id': 3454, 'label': 'B-', 'path': './dataset-C/train/B-/GX013986_20.jpg'}, Distance 0.9536384344100952
Nearest neighbor 3: {'id': 35

In [ ]:

new_birds = ['KR-L', 'X-XX', 'X-XX', 'RM-X', 'RM-X', 'X-XX', '-B', 'K-PW', '-B', 'X-XX',
                'K-PW', 'KR-L', 'KR-L', '-B', 'W-GL', 'X-XX', 'KR-L', 'X-XX', '-B', 'XX-O', 'OL-Y',
                'RM-X', '-B', 'W-GL', 'XX-O', 'W-GL', 'KR-L', 'RM-X', 'RM-X', '-B', 'KR-L', 'X-XX', 'XX-O',
                'W-GL', 'W-GL', 'KR-L', 'RM-X', 'KR-L', 'X-XX', 'X-XX', '-B', 'KR-L', '-B', 'KR-L', 'W-GL',
                'RM-X', 'KR-L', 'W-GL', 'RM-X', '-B', 'W-GL', 'KR-L', '-B', 'X-XX', 'W-GL', 'RM-X', '-B', 'W-GL', 'RM-X',
                '-B', 'KR-L', 'W-GL', 'W-GL', 'XX-O', 'KR-L', 'RM-X', 'RM-X', '-B', '-B', 'X-XX', 'XX-O', '-B', 'W-GL',
                'RM-X', 'OL-Y', 'X-XX', 'X-XX', 'RM-X', '-B', '-B', 'RM-X', 'RM-X', 'RM-X', 'W-GL', 'W-GL',
                'W-GL', 'K-PW', 'RM-X', 'RM-X', 'X-XX', 'KR-L', '-B', 'X-XX', 'W-GL', 'X-XX', 'KR-L', 'KR-L', 'RM-X',
                'W-GL', 'KR-L', 'KR-L', 'OL-Y', 'W-GL', 'KR-L', '-B', 'W-GL', 'KR-L', 'W-GL', 'W-GL', 'X-XX', 'RM-X',
                'X-XX', '-B', 'RM-X', 'W-GL', 'KR-L', 'KR-L', 'W-GL', 'W-GL', 'W-GL', 'KR-L', 'X-XX', 'W-GL', 'W-GL',
                'W-GL', 'KR-L', 'W-GL', '-B', 'W-GL', 'KR-L', 'X-XX', 'X-XX', '-B', 'X-XX', 'KR-L', 'W-GL', 'KR-L',
                'KR-L', 'RM-X', 'X-XX', 'KR-L', 'RM-X', '-B', 'KR-L', '-B', '-B', 'W-GL', 'W-GL', 'X-XX', '-B',
                'X-XX', 'KR-L', 'XX-O', 'X-XX', 'X-XX', 'X-XX', 'KR-L', 'W-GL', '-B', '-B', 'RM-X', '-B', 'X-XX',
                '-B', 'KR-L', '-B', 'KR-L', 'XX-O', 'W-GL', 'KR-L', 'W-GL', 'KR-L', 'W-GL', 'KR-L', '-B', 'W-GL',
                'RM-X', 'RM-X', 'KR-L', 'RM-X', 'W-GL', 'W-GL', '-B', '-B', 'KR-L', 'W-GL', 'X-XX', 'X-XX', '-B',
                'W-GL', 'KR-L', 'X-XX', 'KR-L', 'W-GL', 'RM-X', 'XX-O', 'W-GL', '-B', 'KR-L', 'X-XX', 'XX-O', 'X-XX', 'K-PW', 'RM-X']

print(list(set(new_birds))) #reducing it down to non-repeats
#['X-XX', 'OL-Y', 'K-PW', 'KR-L', 'W-GL', 'XX-O', '-B', 'RM-X']

intersection = list(set(train_generator.class_indices.keys()) & set(C_generator.class_indices.keys()))

print(intersection)

print(len(intersection))




['X-XX', 'OL-Y', 'K-PW', 'KR-L', 'W-GL', 'XX-O', '-B', 'RM-X']
['Y-GL', 'WX-P', 'YM-Y', 'O-LW', 'MO-R', '-Y', 'L-MB', 'B-', 'LM-G']
9
Counter({'B-': 500, 'L-MB': 50, 'YM-Y': 49, 'W-GL': 45, 'WX-P': 42, 'KR-L': 42, 'O-LW': 40, '-B': 36, 'Y-GL': 32, 'X-XX': 32, 'RM-X': 29, 'LM-G': 15, '-Y': 10, 'XX-O': 8, 'K-PW': 3, 'OL-Y': 3})


In [ ]:
all_flagged_birds = ['B-', 'B-', 'Y-GL', 'WX-P', 'B-', 'B-', 'B-', 'KR-L', '-Y', 'L-MB', 'B-', 'X-XX', 'B-', 'B-', 'X-XX', 'L-MB', 'B-', 'YM-Y', 'YM-Y', 'RM-X', 'B-', 'B-', '-Y', 'B-', 'B-', 'RM-X', 'LM-G', 'B-', 'X-XX', 'B-', 'L-MB', 'LM-G', '-B', 'YM-Y', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'K-PW', 'L-MB', 'B-', 'B-', 'YM-Y', 'B-', 'B-', 'YM-Y', '-B', 'B-', 'Y-GL', 'L-MB', 'X-XX', 'B-', 'B-', 'K-PW', 'Y-GL', 'B-', 'WX-P', 'O-LW', 'B-', 'B-', 'L-MB', 'KR-L', 'B-', 'KR-L', 'B-', 'B-', 'Y-GL', '-B', 'O-LW', 'O-LW', 'W-GL', 'O-LW', 'X-XX', 'B-', 'B-', 'B-', 'B-', 'KR-L', 'B-', 'Y-GL', 'B-', 'B-', 'L-MB', 'X-XX', 'B-', 'B-', 'WX-P', 'B-', '-B', 'YM-Y', 'B-', 'XX-O', 'B-', 'B-', 'B-', 'B-', 'WX-P', 'WX-P', 'OL-Y', 'YM-Y', 'RM-X', 'B-', 'WX-P', 'B-', 'B-', 'B-', '-B', 'W-GL', 'B-', 'B-', 'XX-O', 'B-', 'B-', 'B-', 'YM-Y', 'B-', 'B-', 'WX-P', 'B-', 'B-', 'W-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'LM-G', 'KR-L', 'B-', 'B-', 'B-', 'RM-X', 'B-', 'RM-X', 'B-', '-B', 'KR-L', 'B-', 'Y-GL', 'B-', 'X-XX', 'XX-O', 'B-', 'W-GL', 'YM-Y', 'W-GL', 'Y-GL', 'B-', 'KR-L', 'LM-G', 'B-', 'B-', 'YM-Y', 'RM-X', 'B-', 'B-', 'Y-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'L-MB', 'WX-P', 'L-MB', 'B-', 'B-', 'KR-L', 'X-XX', 'X-XX', 'O-LW', 'L-MB', 'B-', 'B-', 'B-', 'L-MB', 'B-', '-B', 'B-', 'B-', 'L-MB', 'KR-L', 'YM-Y', '-B', 'KR-L', 'Y-GL', 'B-', 'B-', 'B-', 'O-LW', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'W-GL', 'B-', 'RM-X', 'B-', 'O-LW', 'B-', 'O-LW', 'B-', 'B-', 'YM-Y', 'B-', 'KR-L', 'W-GL', 'O-LW', 'RM-X', 'B-', '-B', 'Y-GL', 'B-', '-Y', 'B-', 'W-GL', 'Y-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'KR-L', 'B-', 'B-', 'B-', '-B', 'B-', 'X-XX', 'B-', 'B-', 'B-', 'Y-GL', 'B-', 'WX-P', 'B-', 'L-MB', 'W-GL', 'B-', 'RM-X', 'YM-Y', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', '-B', '-Y', 'W-GL', 'L-MB', 'B-', 'RM-X', 'B-', 'L-MB', 'B-', 'L-MB', 'L-MB', 'O-LW', 'B-', 'L-MB', 'L-MB', 'L-MB', 'B-', '-B', 'B-', 'B-', 'B-', 'B-', 'KR-L', 'O-LW', 'B-', 'B-', 'WX-P', 'B-', 'O-LW', 'B-', 'B-', 'B-', 'B-', 'B-', 'O-LW', '-Y', 'B-', 'W-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'L-MB', 'W-GL', 'WX-P', 'B-', 'O-LW', 'XX-O', 'L-MB', 'B-', 'B-', 'B-', 'B-', 'KR-L', 'B-', 'B-', 'RM-X', 'RM-X', 'B-', 'B-', 'O-LW', '-B', 'WX-P', 'B-', 'B-', 'B-', '-B', 'B-', 'B-', 'Y-GL', 'YM-Y', 'B-', 'WX-P', 'B-', 'WX-P', 'B-', 'LM-G', 'B-', 'B-', 'B-', 'X-XX', 'B-', 'O-LW', 'B-', 'WX-P', 'WX-P', 'B-', 'B-', 'B-', 'XX-O', 'B-', 'WX-P', 'B-', 'Y-GL', 'YM-Y', 'B-', 'B-', 'B-', 'B-', '-B', 'B-', 'B-', 'B-', 'B-', 'WX-P', 'W-GL', 'L-MB', 'RM-X', 'B-', 'B-', 'LM-G', 'B-', 'OL-Y', 'B-', 'X-XX', 'B-', 'B-', 'O-LW', 'WX-P', 'X-XX', 'RM-X', 'YM-Y', 'B-', 'O-LW', 'B-', 'B-', '-B', 'B-', 'B-', 'L-MB', '-B', 'B-', 'RM-X', 'B-', 'RM-X', 'RM-X', 'W-GL', 'W-GL', 'B-', 'B-', 'YM-Y', 'B-', 'B-', 'L-MB', 'B-', 'B-', 'YM-Y', 'Y-GL', 'W-GL', 'B-', 'B-', 'B-', 'B-', 'K-PW', 'B-', 'RM-X', 'B-', 'B-', 'B-', 'RM-X', 'B-', 'B-', 'X-XX', 'KR-L', '-B', 'B-', 'B-', 'X-XX', 'W-GL', 'YM-Y', 'B-', 'X-XX', 'B-', 'B-', 'KR-L', 'B-', 'WX-P', 'B-', 'WX-P', 'B-', 'KR-L', 'Y-GL', 'B-', 'B-', 'B-', 'RM-X', 'B-', 'B-', 'O-LW', 'B-', 'B-', 'B-', 'B-', 'W-GL', 'KR-L', 'KR-L', 'B-', 'OL-Y', 'W-GL', 'WX-P', 'KR-L', '-B', 'WX-P', 'B-', 'W-GL', 'L-MB', 'L-MB', 'KR-L', 'B-', 'YM-Y', 'B-', 'B-', 'L-MB', 'B-', 'Y-GL', 'W-GL', 'W-GL', 'B-', 'B-', '-Y', 'B-', 'B-', 'B-', 'X-XX', 'RM-X', 'L-MB', 'X-XX', 'B-', 'WX-P', 'B-', 'YM-Y', '-B', 'B-', 'RM-X', 'B-', 'LM-G', 'W-GL', 'B-', 'B-', 'YM-Y', 'B-', 'Y-GL', 'YM-Y', 'B-', 'B-', 'B-', 'KR-L', 'B-', 'KR-L', 'YM-Y', 'LM-G', '-Y', 'B-', 'YM-Y', 'B-', 'B-', 'O-LW', 'W-GL', 'B-', 'O-LW', 'B-', 'W-GL', 'B-', 'B-', 'B-', 'B-', 'WX-P', 'W-GL', 'B-', 'KR-L', 'B-', 'X-XX', 'O-LW', 'W-GL', 'W-GL', 'B-', 'B-', 'W-GL', 'KR-L', 'B-', 'W-GL', 'L-MB', 'B-', '-B', 'B-', 'O-LW', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'W-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'LM-G', 'B-', 'Y-GL', 'B-', 'B-', 'O-LW', 'B-', 'KR-L', 'WX-P', 'Y-GL', 'O-LW', 'B-', 'L-MB', 'B-', 'WX-P', 'B-', 'B-', 'B-', 'B-', 'B-', 'YM-Y', 'B-', 'B-', 'Y-GL', 'B-', 'X-XX', 'L-MB', 'B-', 'B-', 'Y-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'X-XX', 'WX-P', 'B-', 'WX-P', '-B', 'X-XX', 'B-', 'KR-L', 'B-', 'B-', '-Y', 'W-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'WX-P', 'WX-P', 'YM-Y', 'KR-L', 'B-', 'B-', 'B-', 'B-', 'B-', 'KR-L', 'B-', 'RM-X', 'B-', 'X-XX', 'KR-L', 'L-MB', 'B-', 'RM-X', 'B-', 'B-', 'B-', 'B-', 'B-', 'O-LW', 'O-LW', 'B-', '-Y', 'B-', 'B-', 'YM-Y', '-B', 'B-', 'KR-L', 'B-', 'O-LW', 'Y-GL', 'B-', 'B-', 'B-', 'B-', '-B', 'L-MB', 'B-', 'O-LW', 'LM-G', 'B-', 'YM-Y', 'WX-P', 'L-MB', 'YM-Y', '-B', 'WX-P', 'W-GL', 'B-', 'O-LW', 'B-', 'B-', 'B-', 'B-', 'W-GL', 'X-XX', 'B-', 'L-MB', 'YM-Y', '-B', 'X-XX', 'L-MB', 'B-', 'KR-L', 'B-', 'LM-G', 'B-', 'B-', 'B-', 'XX-O', 'L-MB', 'X-XX', 'YM-Y', 'B-', 'B-', 'X-XX', 'YM-Y', 'B-', 'B-', 'B-', 'YM-Y', 'B-', 'X-XX', 'B-', 'Y-GL', 'B-', 'WX-P', 'B-', 'YM-Y', 'KR-L', 'B-', 'B-', 'B-', 'W-GL', 'YM-Y', 'O-LW', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', '-B', '-B', 'Y-GL', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'RM-X', 'L-MB', 'B-', 'B-', 'YM-Y', 'B-', '-B', 'Y-GL', 'WX-P', 'B-', 'B-', 'O-LW', 'B-', 'L-MB', 'X-XX', 'O-LW', '-B', 'B-', 'KR-L', '-B', 'B-', 'KR-L', 'B-', 'L-MB', 'YM-Y', 'B-', 'B-', 'B-', 'XX-O', 'O-LW', 'B-', 'B-', 'B-', 'O-LW', 'B-', 'W-GL', 'KR-L', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'W-GL', 'B-', 'B-', 'L-MB', 'B-', 'B-', 'L-MB', 'KR-L', 'B-', 'W-GL', 'B-', 'B-', 'KR-L', '-B', 'B-', 'B-', 'W-GL', 'RM-X', 'B-', 'Y-GL', 'B-', 'WX-P', 'WX-P', 'B-', 'B-', 'L-MB', 'B-', 'RM-X', 'B-', 'B-', 'B-', 'KR-L', 'B-', 'YM-Y', 'B-', 'B-', 'B-', 'YM-Y', 'WX-P', 'B-', 'B-', 'B-', 'B-', 'B-', 'O-LW', 'YM-Y', 'B-', 'RM-X', 'YM-Y', 'B-', 'B-', 'B-', 'B-', 'L-MB', 'B-', 'B-', 'O-LW', 'B-', 'B-', 'W-GL', 'B-', 'B-', 'B-', 'WX-P', 'YM-Y', 'B-', 'B-', 'B-', 'B-', 'YM-Y', 'B-', 'B-', 'Y-GL', 'B-', 'B-', 'L-MB', 'W-GL', 'B-', 'B-', 'B-', 'B-', 'YM-Y', 'B-', '-B', 'B-', 'B-', 'B-', 'WX-P', '-B', 'B-', 'KR-L', 'B-', 'B-', 'B-', 'Y-GL', 'W-GL', 'O-LW', 'B-', 'X-XX', 'Y-GL', 'B-', 'LM-G', 'L-MB', 'B-', 'B-', 'X-XX', 'YM-Y', 'L-MB', 'B-', 'LM-G', 'LM-G', 'WX-P', 'YM-Y', 'B-', 'B-', 'B-', '-B', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'B-', 'YM-Y', 'B-', 'W-GL', 'KR-L', 'B-', 'B-', 'B-', 'B-', 'LM-G', 'X-XX', 'KR-L', 'B-', 'B-', 'B-', 'B-', '-Y', 'WX-P', 'W-GL', 'Y-GL', 'O-LW', 'RM-X', 'B-', 'B-', 'Y-GL', 'XX-O', 'O-LW', 'B-', 'W-GL', 'B-', '-B', 'L-MB', 'L-MB', 'B-', 'YM-Y', 'B-', 'B-', 'B-']
print(len(all_flagged_birds))
print(list(set(all_flagged_birds)))

intersection = list(set(train_generator.class_indices.keys()) & set(all_flagged_birds))

dataset_B_set = intersection

count = Counter(all_flagged_birds)
total = 0
for bird in dataset_B_set:
  total += count[bird]

print(f"Dataset B counts: {total}")

dataset_C_set = list(set(new_birds))

count = Counter(all_flagged_birds)
total = 0
for bird in dataset_C_set:
  total += count[bird]

print(f"Dataset C counts: {total}")

936
['Y-GL', 'WX-P', 'X-XX', 'OL-Y', 'K-PW', 'KR-L', 'YM-Y', 'O-LW', 'W-GL', 'XX-O', '-B', 'RM-X', '-Y', 'L-MB', 'B-', 'LM-G']
Dataset B counts: 738
Dataset C counts: 198


In [ ]:
labels = C_generator.classes
counts = Counter(labels)

print(counts)

idx_to_class = {v: k for k, v in C_generator.class_indices.items()}

class_counts = {idx_to_class[idx]: count for idx, count in counts.items()}

total = 0
for bird in dataset_C_set:
  print("")
  total += class_counts[bird]

print(total)

Counter({2: 2097, 1: 735, 12: 440, 16: 434, 4: 316, 11: 235, 5: 211, 15: 160, 0: 112, 10: 111, 13: 105, 8: 103, 6: 91, 3: 28, 14: 20, 7: 13, 9: 7})








934
